# Preprocessing
Notebook pokazuje decyzje przygotowania danych podjęte na podstawie EDA
oraz sprawdza, że kod z `src/preprocessing.py` działa poprawnie.

In [4]:
import sys
sys.path.append("..")

from src.preprocessing import load_data, get_split, build_preprocessor, get_preprocessed_split

## Decyzje preprocessingu

Na podstawie wniosków z EDA podjęliśmy następujące decyzje:

- Brakujące wartości - nic nie robimy, dataset jest kompletny.
- Outliery - zostawiamy, to legalne wartości (np. wiek 92 lat, 4 produkty).
- Usuwamy kolumny RowNumber, CustomerId, Surname - identyfikatory, nic nie wnoszą do modelu.
- Usuwamy kolumnę Complain - jej wartość praktycznie pokrywa się z targetem.
- Skalujemy kolumny numeryczne przez StandardScaler - mają bardzo różne skale.
- Kolumny kategoryczne (Geography, Gender, Card Type) zamieniamy na liczby przez OneHotEncoder.
- Dzielimy dane 80/20 ze stratyfikacją po Exited, żeby w obu zbiorach zachować proporcję klas 80/20.

## Sprawdzenie funkcji load_data

In [5]:
df = load_data()
df.shape, df.columns.tolist()

((10000, 14),
 ['CreditScore',
  'Geography',
  'Gender',
  'Age',
  'Tenure',
  'Balance',
  'NumOfProducts',
  'HasCrCard',
  'IsActiveMember',
  'EstimatedSalary',
  'Exited',
  'Satisfaction Score',
  'Card Type',
  'Point Earned'])

Funkcja działa poprawnie - mamy 10000 wierszy i 14 kolumn (z 18 wyrzucone zostały 4).

## Sprawdzenie funkcji get_split

In [ ]:
X_train, X_test, y_train, y_test, salaries_test = get_split(df)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(8000, 13) (2000, 13)
Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64


Split działa poprawnie - 8000 wierszy w treningowym, 2000 w testowym. Stratyfikacja zachowała proporcje klas: 80% klientów zostaje, 20% odchodzi.

## Sprawdzenie funkcji build_preprocessor

In [4]:
preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train_transformed.shape)
preprocessor.get_feature_names_out()

(8000, 16)


array(['num__CreditScore', 'num__Age', 'num__Tenure', 'num__Balance',
       'num__NumOfProducts', 'num__EstimatedSalary', 'num__Point Earned',
       'num__Satisfaction Score', 'cat__Geography_Germany',
       'cat__Geography_Spain', 'cat__Gender_Male', 'cat__Card Type_GOLD',
       'cat__Card Type_PLATINUM', 'cat__Card Type_SILVER',
       'bin__HasCrCard', 'bin__IsActiveMember'], dtype=object)

Funkcja działa poprawnie. Z 13 kolumn wejściowych dostajemy 16 cech wyjściowych:

- 8 numerycznych,
- 6 kategorycznych po zastosowaniu OneHotEncoding,
- 2 binarne.

## Sprawdzenie funkcji get_preprocessed_split


In [6]:
X_train_t, X_test_t, y_train, y_test, salaries_test, preprocessor = get_preprocessed_split(df)

print(X_train_t.shape)       
print(X_test_t.shape)        
print(len(salaries_test))    
print(salaries_test[:5])      

(8000, 16)
(2000, 16)
2000
[137079.86  27596.39  38152.01  46807.62 105152.17]


Funkcja zwraca gotowe dane do użycia bezpośrednio w notebookach z modelami:
- X_train_t i X_test_t są już po skalowaniu i encodingu (16 cech)
- salaries_test zawiera oryginalne kwoty w dolarach - potrzebne do liczenia zysku banku
